# ResNet-18 CIFAR-10 Teacher vs Linear Students

Load saved probability artifacts for the CIFAR-10 validation split and OOD datasets.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import torch

ROOT = Path.cwd()
if not (ROOT / "runs").exists():
    ROOT = ROOT.parent

EXPERIMENT_NAME = "linear_student_resnet18_cifar10"
PROBABILITY_DIR = ROOT / "runs" / EXPERIMENT_NAME / "probabilities"
MANIFEST_PATH = PROBABILITY_DIR / "manifest.json"

MANIFEST_PATH

In [ ]:
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

manifest["experiment_name"], manifest["checkpoint_selection"]

In [ ]:
def load_probability_artifact(path: str | Path) -> dict[str, object]:
    """Load one saved probability artifact on CPU."""

    artifact_path = Path(path)
    if not artifact_path.is_absolute():
        artifact_path = ROOT / artifact_path
    return torch.load(artifact_path, map_location="cpu", weights_only=False)


artifacts = []
for entry in manifest["artifacts"]:
    artifact = load_probability_artifact(entry["path"])
    artifacts.append({**entry, **artifact})

len(artifacts)

In [ ]:
teacher_probabilities = {
    artifact["dataset"]: artifact
    for artifact in artifacts
    if artifact["model"] == "teacher"
}

student_probabilities = {
    (artifact["dataset"], artifact["method"], artifact["checkpoint"]): artifact
    for artifact in artifacts
    if artifact["model"] == "student"
}

teacher_probabilities.keys(), student_probabilities.keys()

In [ ]:
summary = []
for artifact in artifacts:
    row = {
        "dataset": artifact["dataset"],
        "model": artifact["model"],
        "method": artifact.get("method"),
        "checkpoint": artifact.get("checkpoint"),
        "probabilities_shape": tuple(artifact["probabilities"].shape),
        "labels_shape": tuple(artifact["labels"].shape),
    }
    summary.append(row)

summary

In [ ]:
dataset_name = "cifar10_validation"
method = "mse"
checkpoint = "best"

teacher = teacher_probabilities[dataset_name]
student = student_probabilities[(dataset_name, method, checkpoint)]

teacher_probs = teacher["probabilities"]
student_probs = student["probabilities"]
labels = teacher["labels"]

teacher_probs.shape, student_probs.shape, labels.shape